In [1]:
import numpy as np
import pandas as pd
import os 

# define important funtions

In [1]:
def set_df(file_path):
    # Load the data from the specified file path
    df = pd.read_csv(file_path)
    # Convert the 'datetime' column to datetime format
    df['datetime'] = pd.to_datetime(df['datetime']).dt.strftime('%Y-%m-%d %H:%M:%S')
    # Set 'datetime' as the index
    df.set_index('datetime', inplace=True)
    return df

def kappa_kohler_module(Extra, NSD, pars_temp, switch):
    """
    Calculates cloud condensation nuclei (CCN) concentration and kappa values
    using the kappa-Köhler theory.

    Parameters:
    - Extra: dict
        Additional parameters such as densities and critical supersaturation.
    - NSD: numpy array
        Number size distribution for particles.
    - pars_temp: list or array
        Parameter values for the calculation, which include mass fractions.
    - T: float
        Temperature in Kelvin.
    - SIGMA: float
        Surface tension in N/m.

    Returns:
    - ccn: numpy array
        Calculated cloud condensation nuclei (CCN) concentrations.
    - kappa: float
        Calculated kappa value for the particles.
    - kappa_inorg: float
        Kappa value for inorganic components.
    """
    T = Extra['temp']
    SIGMA = Extra['sigma']
    # Calculate kappa from mass concentration
    if switch == 0:
        kappa = cal_kappa(Extra, pars_temp)  # Total kappa value for particles
    if switch == 1:
        kappa = Extra['kappa_test']
    scrit_kappa = []  # List to store critical supersaturation values for kappa

    # Extract additional parameters from Extra
    dp_dry = Extra['dp_fine_bins']            # Array of dry particle diameters in nm
    ss_amb = Extra['ss_amb']        # Ambient supersaturation levels in %
    d_lower_arr = Extra['d_lower_arr']  # Lower bound of diameter bins in nm
    d_upper_arr = Extra['d_upper_arr']  # Upper bound of diameter bins in nm
    wet_dia = Extra['wet_dia']      # Array of wet particle diameters in nm
    
    # Calculation of critical supersaturation from kappa-Köhler equation
    for dp in dp_dry:
        # Find index of first wet diameter greater than current dry diameter
        index = np.where(wet_dia > dp)[0][0]
        sliced_wet_dia = wet_dia[index:]  # Slice wet diameters to relevant range
        # Calculate supersaturation for each wet diameter
        SS_values = [kappa_kohler(w * 1e-9, dp * 1e-9, kappa, T, SIGMA) for w in sliced_wet_dia]
        # Store the maximum supersaturation value as the critical supersaturation
        scrit_kappa.append((max(SS_values) - 1) * 100)

    # Initialize CCN concentration array
    ccn = np.zeros(len(ss_amb))
    dact_arr = np.zeros(len(ss_amb))
    act_bin_arr = np.zeros(len(ss_amb))
    N_act_bin_arr = np.zeros(len(ss_amb))
    for i, ss in enumerate(ss_amb):
        try:
            act_bin = np.where(np.array(scrit_kappa) < ss)[0][0]

            ccn_x = np.sum(NSD[act_bin + 1:])  # Sum of number size distribution above activation bin
            
            # Calculate the slope for linear interpolation of activation diameter
            slope = (dp_dry[act_bin] - dp_dry[act_bin - 1]) / (scrit_kappa[act_bin] - scrit_kappa[act_bin - 1])
            dact = dp_dry[act_bin - 1] + (ss - scrit_kappa[act_bin - 1]) * slope  # Interpolated activation diameter
    
            # Calculate number of activated particles in the activation bin
            if dact > d_lower_arr[act_bin]:
                N_act_bin = NSD[act_bin] * (d_upper_arr[act_bin] - dact) / (d_upper_arr[act_bin] - d_lower_arr[act_bin])
            elif dact < d_lower_arr[act_bin]:
                multiplier = (d_upper_arr[act_bin - 1] - dact) / (d_upper_arr[act_bin - 1] - d_lower_arr[act_bin - 1])
                N_act_bin = NSD[act_bin] + NSD[act_bin - 1] * multiplier
    
            # Calculate total CCN concentration
            act_bin_arr[i] = int(act_bin)
            N_act_bin_arr[i] =  N_act_bin
            ccn[i] = ccn_x + N_act_bin
            dact_arr[i] = dact
            #print(N_act_bin)
        except:
            act_bin_arr[i] = np.argmax(dp_dry)
            N_act_bin_arr[i] =  0
            ccn[i] = 0
            dact_arr[i] = np.max(dp_dry)

    return act_bin_arr, N_act_bin_arr, ccn, dact_arr, kappa

def cal_kappa(Extra, mass):
    """
    Calculate the kappa hygroscopicity parameter for the particle based on its composition.

    Parameters:
    - Extra: dict
        Additional parameters such as densities and hygroscopicity values.
    - mass: list or array
        Mass concentrations of different particle components.

    Returns:
    - kappa: list
        List containing the total kappa value for particles and the kappa for inorganics.
    """
    # Extract densities from Extra is in kg/m3
    rho_org = Extra['densities'][0]   # Density of organic material
    rho_sulp = Extra['densities'][1]   # Density of ammonium sulfate
    rho_nitr = Extra['densities'][2]  # Density of ammonium nitrate
    rho_bc = Extra['densities'][3]   # Density of black carbon
    
    # Hygroscopicity values for different components
    k_Org = Extra['kappa_org']  # Kappa for organics
    k_N = Extra['kappa_NH4NO3']             # Kappa for ammonium nitrate
    k_s = Extra['kappa_NH4SO4']  
    k_bc = Extra['eBC']  # Kappa for black carbon (assumed non-hygroscopic)

    
    # Calculate total particle volume
    tot_vol = (
        mass[0]/rho_org +  # Volume of organic material
        mass[2]/rho_sulp + # Volume of inorganics
        mass[3]/rho_nitr + # Volume of ammonium nitrate
        mass[1]/rho_bc     # Volume of black carbon
    )

    # Calculate contributions to kappa from each component
    k1 = k_Org * (mass[0]/rho_org) / tot_vol  # Contribution from organics
    k2 = k_s * (mass[2]/rho_sulp) / tot_vol   # Contribution from inorganics
    k3 = k_N * (mass[3]/rho_nitr) / tot_vol   # Contribution from ammonium nitrate (is zero currently)
    k4 = k_bc * (mass[1]/rho_bc) / tot_vol

    # Total kappa value
    kappa = k1 + k2 + k3 + k4

    return kappa

def kappa_kohler(Dwet, Ddry, kappa, T, sigma):
    """
    Calculate the equilibrium supersaturation (s_eq) using the kappa-Köhler equation.

    Parameters:
    - Dwet: float
        Wet particle diameter in meters.
    - Ddry: float
        Dry particle diameter in meters.
    - kappa: float
        Hygroscopicity parameter.
    - T: float
        Temperature in Kelvin.
    - sigma: float
        Surface tension in N/m.

    Returns:
    - s_eq: float
        Equilibrium supersaturation.
    """
    Mw = 18.016 * 1e-3  # Molar mass of water in kg/mol
    R = 8.314           # Universal gas constant in J/(mol*K)
    rhow = 997         # Density of water in kg/m^3

    # Calculate the numerator and denominator of the kappa-Köhler equation
    fact_num = Dwet**3 - Ddry**3
    fact_denum = Dwet**3 - Ddry**3 * (1 - kappa)

    # Calculate the exponential term of the kappa-Köhler equation
    exp_term = np.exp((4 * sigma * Mw) / (R * T * rhow * Dwet))

    # Calculate the equilibrium supersaturation
    s_eq = (fact_num / fact_denum) * exp_term

    return s_eq

def find_new_NSD_act_bin(Extra, unactivated_dact, N, act_bin_idx, switch = None):
    """
    Recalculate the particle number concentration in the activation bin after adjusting the critical diameter.

    This function linearly scales down the number concentration in the activation bin
    based on the new critical activation diameter (`unactivated_dact`), assuming a uniform distribution
    across the bin.

    Parameters:
    - Extra: dict
        Dictionary containing bin edge arrays 'd_lower_arr' and 'd_upper_arr' (in nm).
    - unactivated_dact: float
        Adjusted activation diameter (dact) in nanometers.
    - N: list or array of float
        Number size distribution (NSD) across dry diameter bins.
    - act_bin_idx: int
        Index of the activation bin in the NSD array.

    Returns:
    - N_act_bin_new: float
        Updated number concentration in the activation bin after dact reduction.
    """
    # Extract lower and upper bounds of the diameter bins (in nm)
    d_lower_arr = Extra['d_lower_arr']
    d_upper_arr = Extra['d_upper_arr']
    #print(d_lower_arr)
    if switch == 0:
        # Get bin boundaries for the activation bin
        d_low = d_lower_arr[act_bin_idx]
        d_high = d_upper_arr[act_bin_idx]

        # Linearly scale down the particle count assuming uniform distribution within the bin
        N_act_bin_new = N[act_bin_idx] * (d_high - unactivated_dact) / (d_high - d_low)
    elif switch == 1:
        # Get bin boundaries for the activation bin
        d_low = d_lower_arr[int(act_bin_idx) - 1]
        d_high = d_upper_arr[int(act_bin_idx) - 1]
        N_act_bin_new = N[int(act_bin_idx - 1)] * (d_high - unactivated_dact) / (d_high - d_low)

    return N_act_bin_new


import numpy as np
from scipy.optimize import brentq

def cal_LWC_correct(particle_number, ddry, dwet):
    """
    Function to calculate LWC based on the difference between wet and dry droplet volumes.
    
    Parameters:
    particle_number : list
        Number of particles per bin.
    ddry : list
        Corresponding dry diameters of the particles (nm).
    dwet : list
        Corresponding wet diameters of the particles (nm).
    
    Returns:
    LWC : float
        Calculated Liquid Water Content (kg/m^3).
    """
    rho_w = 997  # Density of water (kg/m^3)
    
    # Convert diameters to radii (meters)
    radius_dry = np.array(ddry) / 2 * 1e-9  # Convert nm to meters
    radius_wet = np.array(dwet) / 2 * 1e-9  # Convert nm to meters
    
    # Compute volume of each droplet (wet and dry)
    volume_dry = (4/3) * np.pi * radius_dry**3  # m^3
    volume_wet = (4/3) * np.pi * radius_wet**3  # m^3
    
    # Compute mass of liquid water in each bin
    mass_water = (volume_wet - volume_dry) * rho_w  # kg
    
    # Compute total LWC (multiply mass of water in each bin by corresponding particle number in per cubic meter)
    LWC = np.sum(particle_number*1e6 * mass_water)  # kg/m^3
    
    return LWC


def kappa_kohler_for_Dwet(Dwet, Ddry, kappa, T, sigma):
    """
    Calculate the equilibrium supersaturation (s_eq) using the kappa-Köhler equation.

    Parameters:
    - Dwet: float
        Wet particle diameter in meters.
    - Ddry: float
        Dry particle diameter in meters.
    - kappa: float
        Hygroscopicity parameter.
    - T: float
        Temperature in Kelvin.
    - sigma: float
        Surface tension in N/m.

    Returns:
    - s_eq: float
        Equilibrium supersaturation.
    """
    Mw = 18.016 * 1e-3  # Molar mass of water in kg/mol
    R = 8.314           # Universal gas constant in J/(mol*K)
    rhow = 997         # Density of water in kg/m^3

    # Calculate the numerator and denominator of the kappa-Köhler equation
    fact_num = Dwet**3 - Ddry**3
    fact_denum = Dwet**3 - Ddry**3 * (1 - kappa)

    # Calculate the exponential term of the kappa-Köhler equation
    exp_term = np.exp((4 * sigma * Mw) / (R * T * rhow * Dwet))

    # Calculate the equilibrium supersaturation
    s_eq = (fact_num / fact_denum) * exp_term

    return s_eq 

def estimate_dp_wet(Extra, kappa_val, ddry_ls):
    """
    Estimate the critical wet diameter (Dwet_max) at the peak supersaturation.

    Parameters:
    - Extra: dict
        Dictionary containing temperature ('temp') in Kelvin and surface tension ('sigma') in N/m.
    - kappa_val: float
        Hygroscopicity parameter.
    - SS_target: float
        Target supersaturation.
    - ddry_ls: list of float
        List of dry particle diameters in nm.

    Returns:
    - Dwet_at_max_SS: list of float
        List of critical wet diameters (Dwet_max) in nm for each dry diameter.
    """
    T = Extra['temp']  # Temperature in Kelvin
    SIGMA = Extra['sigma']  # Surface tension in N/m
    wet_dia = Extra['wet_dia']
    dp_dry = ddry_ls
    
    Dwet_at_max_SS = []
    for dp in dp_dry:
        index = np.where(wet_dia > dp)[0][0]
        
        sliced_wet_dia = wet_dia[index:]  # Slice wet diameters to relevant range
        #print(sliced_wet_dia)
        SS_values = [kappa_kohler_for_Dwet(w * 1e-9, dp * 1e-9, kappa_val, T, SIGMA) for w in sliced_wet_dia]
        SS_values_percent = [(ss - 1) * 100 for ss in SS_values]
        #print(SS_values_percent)
        peak_idx = np.argmax(SS_values_percent)
        #print(peak_idx)
        Dwet_crit = sliced_wet_dia[peak_idx]
        Dwet_at_max_SS.append(Dwet_crit)
    
    return Dwet_at_max_SS

def solve_dwet(Extra, SS_percentage, kappa, Ddry_list, Dwet_max_list):
    """
    Solve for the wet particle diameter (Dwet) for a list of dry diameters using the kappa-Köhler equation.

    Parameters:
    - s_eq: float
        Equilibrium supersaturation.
    - Ddry_list: list of float
        List of dry particle diameters in meters.
    - kappa: float
        Hygroscopicity parameter.
    - T: float
        Temperature in Kelvin.
    - sigma: float
        Surface tension in N/m.
    - Dwet_max_list: list of float
        List of maximum wet particle diameters in nm (from estimate_dp_wet function).

    Returns:
    - Dwet_solutions: list of float
        List of wet particle diameters in meters corresponding to each dry diameter.
    """
    Mw = 18.016e-3  # Molar mass of water in kg/mol
    R = 8.314       # Universal gas constant in J/(mol*K)
    rhow = 997     # Density of water in kg/m^3
    
    T = Extra['temp']  # Temperature in Kelvin
    SIGMA = Extra['sigma']  # Surface tension in N/m
    wet_dia = Extra['wet_dia']
    
    Dwet_solutions = []
        
    s_eq = 1 + (SS_percentage / 100)
    
    for Ddry, Dwet_max in zip(Ddry_list, Dwet_max_list):
        
        def equation(Dwet):
            if Dwet <= Ddry_m:
                return 1e6  # Avoid nonphysical solutions
            
            fact_num = Dwet**3 - Ddry_m**3
            fact_denum = Dwet**3 - Ddry_m**3 * (1 - kappa)
            exp_term = np.exp((4 * SIGMA * Mw) / (R * T * rhow * Dwet))
            
            return (fact_num / fact_denum) * exp_term - s_eq
        
    
        # Convert inputs to SI units (meters)
        Ddry_m = Ddry * 1e-9
        Dwet_max_m = Dwet_max * 1e-9
        try:
            val = equation(Ddry_m * 1.01)
            if np.isnan(val):
                print("NaN detected at lower bound:", Ddry_m * 1.01)
        except Exception as e:
            print("Error evaluating equation:", e)
        #print(Ddry_m*1.01*1e9 , Dwet_max_m*1e9)
        #f_a = equation(Ddry_m * 1.01)
        #f_b = equation(Dwet_max_m)
        
        Dwet_solution = brentq(equation, Ddry_m * 1.01, Dwet_max_m)
        Dwet_solutions.append(Dwet_solution * 1e9)
        
    return Dwet_solutions

def solve_dwet_single(Extra, SS_percentage, kappa, Ddry, Dwet_max):
    """
    Solve for the wet particle diameter (Dwet) using the kappa-Köhler equation for a single dry diameter.

    Parameters:
    - Extra: dict
        Dictionary containing physical constants like temperature and surface tension.
    - SS_percentage: float
        Ambient supersaturation (%) — e.g., 0.3 for 0.3% SS.
    - kappa: float
        Hygroscopicity parameter.
    - Ddry: float
        Dry particle diameter in nanometers.
    - Dwet_max: float
        Maximum wet particle diameter in nanometers.

    Returns:
    - Dwet_solution: float or np.nan
        Wet diameter in nanometers if root is found, otherwise np.nan.
    """
    import numpy as np
    from scipy.optimize import brentq

    Mw = 18.016e-3  # kg/mol
    R = 8.314       # J/(mol*K)
    rhow = 997     # kg/m^3

    T = Extra['temp']      # K
    SIGMA = Extra['sigma'] # N/m

    Ddry_m = Ddry * 1e-9       # Convert to meters
    Dwet_max_m = Dwet_max * 1e-9
    s_eq = 1 + (SS_percentage / 100)

    def equation(Dwet):
        if Dwet <= Ddry_m:
            return 1e6  # nonphysical region
        fact_num = Dwet**3 - Ddry_m**3
        fact_denum = Dwet**3 - Ddry_m**3 * (1 - kappa)
        exp_term = np.exp((4 * SIGMA * Mw) / (R * T * rhow * Dwet))
        return (fact_num / fact_denum) * exp_term - s_eq

    # Evaluate function at bounds to check sign change
    try:
        f_a = equation(Ddry_m * 1.01)
        f_b = equation(Dwet_max_m)
    except Exception as e:
        print("Error evaluating function:", e)
        return np.nan
 
    Dwet_solution = brentq(equation, Ddry_m * 1.01, Dwet_max_m)
    return Dwet_solution * 1e9  # return in nanometers


def estimate_dp_wet_single(Extra, kappa_val, ddry):
    """
    Estimate the critical wet diameter (Dwet_max) at the peak supersaturation for a single dry diameter.

    Parameters:
    - Extra: dict
        Dictionary containing temperature ('temp') in Kelvin and surface tension ('sigma') in N/m.
    - kappa_val: float
        Hygroscopicity parameter.
    - ddry: float
        Dry particle diameter in nm.

    Returns:
    - Dwet_crit: float
        Critical wet diameter (Dwet_max) in nm.
    """
    T = Extra['temp']        # Temperature in Kelvin
    SIGMA = Extra['sigma']   # Surface tension in N/m
    wet_dia = Extra['wet_dia']  # Array of wet diameters (nm)

    # Find index where wet_dia just exceeds ddry
    try:
        index = np.where(wet_dia > ddry)[0][0]
    except IndexError:
        print(f"⚠️ No wet diameter larger than dry diameter {ddry} nm found.")
        return np.nan

    sliced_wet_dia = wet_dia[index:]

    # Compute supersaturation values using Köhler theory
    SS_values = [kappa_kohler_for_Dwet(w * 1e-9, ddry * 1e-9, kappa_val, T, SIGMA) for w in sliced_wet_dia]
    SS_values_percent = [(ss - 1) * 100 for ss in SS_values]

    # Find peak supersaturation
    peak_idx = np.argmax(SS_values_percent)
    Dwet_crit = sliced_wet_dia[peak_idx]

    return Dwet_crit




In [7]:
import pandas as pd

def get_fog_data(fog_event_number):
    """
    Extract NSD_fog and AMS_fog data for a given fog event number.
    
    Parameters:
        fog_event_number (int): Row number of the fog event in fog_dates.
    
    Returns:
        NSD_fog (DataFrame): Data for NSD during the fog event.
        AMS_fog (DataFrame): Data for AMS within ±10 minutes of NSD_fog timestamps.
    """
    # Load fog event data
    fog_dates = pd.read_excel('../input_data/fog-times.xlsx')
    fog_dates.columns = ['fog_nr', 'start_time', 'end_time']  
    
    # Convert timestamps to datetime and localize to Rome timezone
    for col in ['start_time', 'end_time']:
        fog_dates[col] = pd.to_datetime(fog_dates[col]).dt.tz_localize('UTC').dt.tz_convert('Europe/Rome')
        fog_dates[col] = fog_dates[col].dt.strftime('%Y-%m-%d %H:%M:%S')
    
    # Select the specified fog event
    fog_event = fog_dates.iloc[[fog_event_number]]
    
    # Function to check if a timestamp falls within the fog event
    def is_fog_present(timestamp, fog_df):
        return any((timestamp >= fog_event['start_time']) and (timestamp <= fog_event['end_time']) for _, fog_event in fog_df.iterrows())
    
    # Load NSD data
    NSD_fitted = pd.read_csv('../input_data/Bimodal_parameters_test_new.csv')
    NSD_fitted['datetime'] = pd.to_datetime(NSD_fitted['datetime'])
    NSD_fitted_filtered = NSD_fitted[NSD_fitted['mode1_n'] >= 1e-10].copy()  # Create a copy explicitly
    NSD_fitted_filtered['datetime'] = NSD_fitted_filtered['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    NSD_fitted_filtered.set_index('datetime', inplace=True)
    
    NSD_mode1_scaled = set_df('../input_data/NSD_scaled_mode1_timeseries.csv')
    NSD_mode2_scaled = set_df('../input_data/NSD_scaled_mode2_timeseries.csv')
    
    # Calculate the row sums and retain only the sum column
    NSD_mode1_scaled['mode1_conc'] = NSD_mode1_scaled.sum(axis=1)
    NSD_mode1_scaled = NSD_mode1_scaled[['mode1_conc']]
    
    NSD_mode2_scaled['mode2_conc'] = NSD_mode2_scaled.sum(axis=1)
    NSD_mode2_scaled = NSD_mode2_scaled[['mode2_conc']]
    # Combine the two dataframes into one
    NSD_combined = pd.concat([NSD_mode1_scaled, NSD_mode2_scaled], axis=1)
    # Concatenate the harmonized dataframes
    NSD_final = pd.concat([NSD_fitted_filtered, NSD_combined], axis = 1).dropna()
    NSD_final_new = NSD_final.iloc[:,[0,1,6,3,4,7]]

    NSD_final_new_1 = NSD_final_new.reset_index()
    NSD_final_new_1['is_fog'] = NSD_final_new_1['datetime'].apply(lambda x: is_fog_present(x, fog_event))
    NSD_fog = NSD_final_new_1[NSD_final_new_1['is_fog']].drop(columns=['is_fog']).set_index('datetime')
    
    # Load AMS data
    AMS = set_df('../input_data/Kappa_timeseries_with_new_BC_less_data_gaps.csv')
    
    # Ensure datetime indices are in proper format
    AMS.index = pd.to_datetime(AMS.index)
    NSD_fog.index = pd.to_datetime(NSD_fog.index)
    
    # Define a ±10-minute time window for AMS and NSD observation in vicinity of fog
    time_window_before = pd.Timedelta(minutes = 30)
    time_window_after = pd.Timedelta(minutes = 20)
    
    # Find AMS rows within ±10 minutes of NSD_fog timestamps
    mask = AMS.index.to_series().apply(lambda x: any((NSD_fog.index >= x - time_window_before) & (NSD_fog.index <= x + time_window_after)))
    
    # Filter AMS data
    AMS_fog = AMS[mask]
 
    return NSD_fog, AMS_fog

# load data

In [8]:
# Load fog event data
fog_dates = pd.read_excel('../input_data/fog-times.xlsx')
fog_dates.columns = ['fog_nr', 'start_time', 'end_time']  

# Convert timestamps to datetime and localize to Rome timezone
for col in ['start_time', 'end_time']:
    fog_dates[col] = pd.to_datetime(fog_dates[col]).dt.tz_localize('UTC').dt.tz_convert('Europe/Rome')
    #fog_dates[col] = fog_dates[col].dt.strftime('%Y-%m-%d %H:%M:%S')

# Load NSD data
NSD_fitted = pd.read_csv('../input_data/Bimodal_parameters_test_new.csv')
NSD_fitted['datetime'] = pd.to_datetime(NSD_fitted['datetime'])
NSD_fitted_filtered = NSD_fitted[NSD_fitted['mode1_n'] >= 1e-10].copy()  # Create a copy explicitly
NSD_fitted_filtered['datetime'] = NSD_fitted_filtered['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
NSD_fitted_filtered.set_index('datetime', inplace=True)

# scaled number concentrations
NSD_mode1_scaled = set_df('../input_data/NSD_scaled_mode1_timeseries.csv')
NSD_mode2_scaled = set_df('../input_data/NSD_scaled_mode2_timeseries.csv')

# Calculate the row sums and retain only the sum column
NSD_mode1_scaled['mode1_conc'] = NSD_mode1_scaled.sum(axis=1)
NSD_mode1_scaled = NSD_mode1_scaled[['mode1_conc']]

NSD_mode2_scaled['mode2_conc'] = NSD_mode2_scaled.sum(axis=1)
NSD_mode2_scaled = NSD_mode2_scaled[['mode2_conc']]

# Combine the two dataframes into one
NSD_combined = pd.concat([NSD_mode1_scaled, NSD_mode2_scaled], axis = 1)

# Concatenate the harmonized dataframes
NSD_final = pd.concat([NSD_fitted_filtered, NSD_combined], axis = 1).dropna()
NSD_final_new = NSD_final.iloc[:,[0,1,6,3,4,7]]
NSD_final_new.index = pd.to_datetime(NSD_final_new.index, utc = True).tz_convert('Europe/Rome')

# Load AMS data
AMS = pd.read_csv('../input_data/Kappa_timeseries_with_new_BC_less_data_gaps.csv')
AMS['datetime'] = pd.to_datetime(AMS['datetime'])
AMS.set_index('datetime', inplace = True)
AMS.index = pd.to_datetime(AMS.index, utc=True).tz_convert('Europe/Rome')

# Function to check if a timestamp falls within any fog period
def is_fog_present(timestamp, fog_periods):
    for _, row in fog_periods.iterrows():
        if row['start_time'] <= timestamp <= row['end_time']:
            return 1
    return 0

# Ensure datetime index is timezone-aware
NSD_final_new = NSD_final_new.copy()
NSD_final_new.index = pd.to_datetime(NSD_final_new.index)
NSD_final_new['fog_present'] = NSD_final_new.index.to_series().apply(lambda x: is_fog_present(x, fog_dates))
NSD_final_new_fog_id = NSD_final_new.reset_index()
NSD_final_new_fog_id['datetime'] = NSD_final_new_fog_id['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
NSD_final_new_fog_id.set_index('datetime', inplace=True)

AMS.index = pd.to_datetime(AMS.index)
AMS['fog_present'] = AMS.index.to_series().apply(lambda x: is_fog_present(x, fog_dates))
AMS_fog_id = AMS.reset_index()
AMS_fog_id['datetime'] = AMS_fog_id['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
AMS_fog_id.set_index('datetime', inplace=True)

In [9]:
NSD_fog = NSD_final_new_fog_id[NSD_final_new_fog_id.fog_present == 1]
AMS_fog = AMS_fog_id[AMS_fog_id.fog_present == 1]

# Compute percentiles for all numeric columns in AMS_fog
percentiles_AMS = AMS_fog.dropna().quantile([0.05, 0.25, 0.5, 0.75, 0.95])
percentiles_NSD = NSD_fog.quantile([0.05, 0.25, 0.5, 0.75, 0.95])

# Print the percentiles
print("AMS Fog Event Percentiles:")
print(percentiles_AMS)

# Print the percentiles
print("NSD Fog Event Percentiles:")
print(percentiles_NSD)

AMS Fog Event Percentiles:
            Org        BC    NH4SO4     NH4NO3     kappa  fog_present
0.05   0.397769  0.051874  0.205685   0.879296  0.348522          1.0
0.25   1.087820  0.159915  0.515647   2.377768  0.404412          1.0
0.50   2.345820  0.337848  0.966664   4.469885  0.452537          1.0
0.75   5.487550  0.907616  1.997658   8.145950  0.478173          1.0
0.95  11.670875  2.253450  3.742161  20.088663  0.514229          1.0
NSD Fog Event Percentiles:
        mode1_d  mode1_sigma   mode1_conc     mode2_d  mode2_sigma  \
0.05  36.521634     1.366667   610.261765  106.216983     1.566667   
0.25  42.828418     1.500000  1280.349201  124.509713     1.666667   
0.50  48.535626     1.633333  1939.281112  134.970101     1.733333   
0.75  61.882228     1.733333  2891.320876  160.208499     1.766667   
0.95  77.570569     1.866667  4127.933405  199.098608     1.900000   

       mode2_conc  fog_present  
0.05  1714.649285          1.0  
0.25  2670.023068          1.0  
0.50  

# calculate LWC of hydrated aerosols with truth being median of all fog event; SS vs N2; SS and N2 varying between almost entire range of N2 values during the fog events

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cal_NSD as CN  # Import your custom size distribution module
import os
import numpy as np

# Plot style settings
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.weight"] = "bold"
plt.rcParams["font.size"] = 12

# Set number of samples; can be increased 
num_fog_events = 21
num_NSD = 30  
n_SS = 50

# Input directory
obs_dir = '../input_data/'
Extra = {}

# Load dry particle diameter (dp)
with open(os.path.join(obs_dir, 'Dp.txt'), 'r') as file_dp_open:
    dp_dry = np.array([float(val) for val in file_dp_open.read().split()])
Extra['dp'] = dp_dry

# Bin boundaries for dp_dry
#d_upper0 = dp_dry[0] + (dp_dry[1] - dp_dry[0]) / 2
#d_lower0 = dp_dry[0] - (dp_dry[1] - dp_dry[0]) / 2
#d_lower_arr_vec = np.full_like(dp_dry, d_lower0)
#d_upper_arr_vec = np.full_like(dp_dry, d_upper0)

#for i in range(1, len(dp_dry)):
#    d_lower_arr_vec[i] = d_upper_arr_vec[i - 1]
#    d_upper_arr_vec[i] = dp_dry[i] + dp_dry[i] - d_lower_arr_vec[i]

#Extra['d_lower_arr'] = d_lower_arr_vec
#Extra['d_upper_arr'] = d_upper_arr_vec


# Use min and max from dp_dry to set the range
dp_min = dp_dry.min()
dp_max = dp_dry.max()

# Create 400 logarithmically spaced bins
dp_bins = np.logspace(np.log10(dp_min), np.log10(dp_max), 2000)

# dp_bins: 400 log-spaced bin centers (already created)
d_upper0 = dp_bins[0] + (dp_bins[1] - dp_bins[0]) / 2
d_lower0 = dp_bins[0] - (dp_bins[1] - dp_bins[0]) / 2
d_lower_arr_vec = np.full_like(dp_bins, d_lower0)
d_upper_arr_vec = np.full_like(dp_bins, d_upper0)

for i in range(1, len(dp_bins)):
    d_lower_arr_vec[i] = d_upper_arr_vec[i - 1]
    d_upper_arr_vec[i] = dp_bins[i] + dp_bins[i] - d_lower_arr_vec[i]

Extra['d_lower_arr'] = d_lower_arr_vec
Extra['d_upper_arr'] = d_upper_arr_vec

# Optionally store in Extra
Extra['dp_fine_bins'] = dp_bins

# Physical constants
Extra['densities'] = [1.50e3, 1.77e3, 1.71e3, 1.77e3]
Extra['sigma'] = 72.8e-3
Extra['wet_dia'] = np.logspace(0, 6.35, 3000)
Extra['temp'] = 273.15 + 19.9
Extra['kappa_org'] = 0.1
Extra['kappa_NH4SO4'] = 0.61
Extra['kappa_NH4NO3'] = 0.67
Extra['eBC'] = 0


# Containers for all events
all_NSD = []
all_AMS = []

# Load and aggregate data
for i in range(num_fog_events):
    NSD_fog, AMS_fog = get_fog_data(i)
    if NSD_fog is not None and AMS_fog is not None and not NSD_fog.empty and not AMS_fog.empty:
        all_NSD.append(NSD_fog.dropna())
        all_AMS.append(AMS_fog.dropna())

# Combine all fog events' NSD and AMS data
combined_NSD = pd.concat(all_NSD, axis=0)
combined_AMS = pd.concat(all_AMS, axis=0)

# Compute percentiles
percentiles_NSD = combined_NSD.quantile([0.01, 0.5, 1])
percentiles_AMS = combined_AMS.quantile([0.01, 0.5, 1])
print(percentiles_NSD)
mode2_conc_values = np.linspace(percentiles_NSD.mode2_conc.min(), 
                                percentiles_NSD.mode2_conc.max(), num_NSD)

#Extra['ss_amb'] = np.logspace(np.log10(0.005), np.log10(1.2), n_SS)
Extra['ss_amb'] = np.logspace(np.log10(0.006), np.log10(0.1), n_SS)
# Use median NSD parameters only
params = percentiles_NSD.iloc[1]  # Mid percentile
N1, D1, GSD1 = params[2], params[0], params[1]
D2, GSD2 = params[3], params[4]
comp = percentiles_AMS.iloc[1].values.tolist()
Extra['kappa_test'] = percentiles_AMS.kappa.iloc[1]

# Prepare empty LWC array
LWC_hydrated = np.zeros((len(mode2_conc_values), n_SS))
nsd_samples = []
# Main loop
for j in range(len(mode2_conc_values)):
#for j in range(1):
    print(f"Processing: N2={mode2_conc_values[j]:.2f} ({j+1}/{len(mode2_conc_values)})")
    N2 = mode2_conc_values[j]
    modes = np.array([[N1, GSD1, D1], [N2, GSD2, D2]])

    nsd_lognorm_net, nsd_abs_net = CN.size_distribution(modes, Extra['dp_fine_bins'])
    #print(nsd_lognorm_net)
    nsd_samples.append(nsd_lognorm_net)
    
    act_bin_, N_act_bin_, pred_ccn, dact, kappa = kappa_kohler_module(Extra, nsd_abs_net, comp, 1)

    for SS in range(n_SS):
    #for SS in range(1):
        LWC_hydrated[j, SS] = 0
        if dact[SS] == 792.:
            unactivated_NSD = nsd_abs_net
            unactivated_ddry = dp_bins 

            Dcrits = estimate_dp_wet(Extra, kappa, unactivated_ddry)
            eq_dwet_unactivated = solve_dwet(Extra, Extra['ss_amb'][SS], kappa, unactivated_ddry, Dcrits)
            LWC_hydrated[j, SS] = cal_LWC_correct(unactivated_NSD, unactivated_ddry, eq_dwet_unactivated)
         
        elif d_lower_arr_vec[int(act_bin_[SS])] < dact[SS] < 792.:
            
            unactivated_NSD = nsd_abs_net[:int(act_bin_[SS] - 1)]
            unactivated_ddry = dp_bins[:int(act_bin_[SS] - 1)]
            unactivated_NSD_act_bin = [N_act_bin_[SS]]
            
            Dcrits = estimate_dp_wet(Extra, kappa, unactivated_ddry)
            eq_dwet_unactivated = solve_dwet(Extra, Extra['ss_amb'][SS], kappa, unactivated_ddry, Dcrits)
            LWC_hydrated_ = cal_LWC_correct(unactivated_NSD, unactivated_ddry, eq_dwet_unactivated)
            #print(LWC_hydrated_)
            # Solve for critical particle, retry if brentq fails
            original_dact = dact[SS]
            unactivated_dact = original_dact  # working copy
            #print('initial Dact', unactivated_dact)
            max_attempts = 100
            attempt = 0
            success = False
            while attempt < max_attempts and not success:
                try:
                    Dcrits_dact = estimate_dp_wet_single(Extra, kappa, unactivated_dact)
                    eq_dwet_dact = solve_dwet_single(Extra, Extra['ss_amb'][SS], kappa, unactivated_dact, Dcrits_dact)
                    success = True
                except ValueError as e:
                    #print(f"Retry {attempt+1}: brentq failed for dact={unactivated_dact:.2f} nm — {e}")
                    unactivated_dact -= 0.1  # decrement by 0.1 nm
                    attempt += 1
            if not success:
                print(f"Could not solve for dact = {dact[SS]} nm at SS index {SS}")
                eq_dwet_dact = [np.nan]
            else: 
                new_NSD_act_bin = find_new_NSD_act_bin(Extra, unactivated_dact, nsd_abs_net, int(act_bin_[SS]), switch = 0)
                #print(int(act_bin_[SS]), unactivated_dact)
                LWC_hydrated_dact = cal_LWC_correct(new_NSD_act_bin, unactivated_dact, eq_dwet_dact)
                LWC_hydrated[j, SS] = LWC_hydrated_ + LWC_hydrated_dact
                #print(LWC_hydrated_dact)
        
        elif d_lower_arr_vec[int(act_bin_[SS])] > dact[SS] < 792.:
            
            unactivated_NSD = nsd_abs_net[:int(act_bin_[SS] - 2)]
            unactivated_ddry = dp_bins[:int(act_bin_[SS] - 2)] ##
            unactivated_NSD_act_bin =  [N_act_bin_[SS]]
            unactivated_dact = [dact[SS]]

            Dcrits = estimate_dp_wet(Extra, kappa, unactivated_ddry)
            eq_dwet_unactivated = solve_dwet(Extra, Extra['ss_amb'][SS], kappa, unactivated_ddry, Dcrits)
            LWC_hydrated_ = cal_LWC_correct(unactivated_NSD, unactivated_ddry, eq_dwet_unactivated)

    
            # Solve for critical diameter, retry if brentq fails
            original_dact = dact[SS]
            unactivated_dact = original_dact  # working copy
            #print('initial Dact', unactivated_dact)
            max_attempts = 100
            attempt = 0
            success = False
            while attempt < max_attempts and not success:
                try:
                    Dcrits_dact = estimate_dp_wet_single(Extra, kappa, unactivated_dact)
                    eq_dwet_dact = solve_dwet_single(Extra, Extra['ss_amb'][SS], kappa, unactivated_dact, Dcrits_dact)
                    success = True
                except ValueError as e:
                    #print(f"Retry {attempt+1}: brentq failed for dact={unactivated_dact:.2f} nm — {e}")
                    unactivated_dact -= 0.1  # decrement by 0.1 nm
                    attempt += 1
            if not success:
                print(f"Could not solve for dact = {dact[SS]} nm at SS index {SS}")
                eq_dwet_dact = [np.nan]
            else:
                #unactivated_dact = unactivated_dact  # wrap in list for consistency
                #print('After Dact', unactivated_dact)
                new_NSD_act_bin = find_new_NSD_act_bin(Extra, unactivated_dact, nsd_abs_net, int(act_bin_[SS]), switch = 1)
                LWC_hydrated_dact = cal_LWC_correct(new_NSD_act_bin, unactivated_dact, eq_dwet_dact)
                LWC_hydrated[j, SS] = LWC_hydrated_ + LWC_hydrated_dact

# Flatten and save

N2_vals = np.repeat(mode2_conc_values, n_SS)
SS_vals = np.tile(Extra['ss_amb'], len(mode2_conc_values))
LWC_vals = LWC_hydrated.flatten()

df = pd.DataFrame({
    'N2': N2_vals,
    'SS': SS_vals,
    'LWC': LWC_vals
})

# Format column names as c{diameter in nm}c
column_names = [f"c{d*1e9:.2f}c" for d in Extra['dp_fine_bins']]

# Create DataFrame with formatted column names
nsd_df_explored = pd.DataFrame(nsd_samples, columns=column_names)

df.to_csv("new_results/LWC_sensitivity_all_events_SS_range_interpolated.csv", index = False)
nsd_df_explored.to_csv('new_results/NSD_explored.csv')

         mode1_d  mode1_sigma   mode1_conc     mode2_d  mode2_sigma  \
0.01   31.618384     1.300000   433.130258  102.180751     1.467667   
0.50   50.224298     1.633333  1973.796502  136.063303     1.733333   
1.00  113.234773     2.066667  5992.541130  373.402536     2.033333   

       mode2_conc  
0.01   682.179004  
0.50  3934.986913  
1.00  8309.372675  
Processing: N2=682.18 (1/30)
Processing: N2=945.19 (2/30)
Processing: N2=1208.19 (3/30)
Processing: N2=1471.20 (4/30)
Processing: N2=1734.21 (5/30)
Processing: N2=1997.21 (6/30)
Processing: N2=2260.22 (7/30)
Processing: N2=2523.23 (8/30)
Processing: N2=2786.23 (9/30)
Processing: N2=3049.24 (10/30)
Processing: N2=3312.25 (11/30)
Processing: N2=3575.25 (12/30)
Processing: N2=3838.26 (13/30)
Processing: N2=4101.27 (14/30)
Processing: N2=4364.27 (15/30)
Processing: N2=4627.28 (16/30)
Processing: N2=4890.29 (17/30)
Processing: N2=5153.29 (18/30)
Processing: N2=5416.30 (19/30)
Processing: N2=5679.31 (20/30)
Processing: N2=5942.31 (21

In [11]:
# Containers for all events
all_NSD = []
all_AMS = []

# Load and aggregate data
for i in range(num_fog_events):
    NSD_fog, AMS_fog = get_fog_data(i)
    if NSD_fog is not None and AMS_fog is not None and not NSD_fog.empty and not AMS_fog.empty:
        all_NSD.append(NSD_fog.dropna())
        all_AMS.append(AMS_fog.dropna())

# Combine all fog events' NSD and AMS data
combined_NSD = pd.concat(all_NSD, axis=0)
combined_AMS = pd.concat(all_AMS, axis=0)

# Compute percentiles
percentiles_NSD = combined_NSD.quantile([0.05, 0.5, 0.95])
percentiles_AMS = combined_AMS.quantile([0.05, 0.5, 0.95])
print(percentiles_NSD)

        mode1_d  mode1_sigma   mode1_conc     mode2_d  mode2_sigma  \
0.05  36.521634     1.366667   610.261765  106.386437     1.566667   
0.50  50.224298     1.633333  1973.796502  136.063303     1.733333   
0.95  79.877461     1.866667  4134.540163  206.768140     1.895000   

       mode2_conc  
0.05  1268.893838  
0.50  3934.986913  
0.95  6932.938492  


# save the median values of each event

In [29]:
# Create a DataFrame with the values
median_values = pd.DataFrame({
    'Parameter': ['kappa_median', 'SS_median', 'N2_median'],
    'Value': [median_kappa, SS_median, median_N2]
})

# Save to CSV
median_values.to_csv("new_results/Sensitivity_data_for_paper/median_values.csv", index=False)